# Feature Engineering - NO2

En este notebook construimos las features nuevas a partir del dataset final (`train.parquet`), 
pensadas para sacarle mas partido al modelo de regresion que prediga episodios de alta contaminacion.

**Que vamos a hacer:**
1. Cargar los datos desde `../data_sample/`
2. Split cronologico train_fe / test_fe (sin tocar el train/test del notebook 02)
3. Features temporales (ciclicas, estacion del anio, finde)
4. Lags y medias moviles de NO2 por estacion
5. Imputacion de meteo con flag de missing
6. Target encoding de estacion
7. Pipeline completo: fit en train_fe, transform en test_fe

Trabajamos siempre evitando fuga de datos (data leakage): todo lo que "aprende" de los datos 
(medianas, medias por estacion) se calcula solo con `train_fe`, nunca con `test_fe`.

## 1. Imports

Librerias necesarias para cargar los datos y montar el pipeline de feature engineering.

In [1]:
# Imports generales + para el pipeline de feature engineering
import glob
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin  # clases base para crear transformers custom compatibles con sklearn
from sklearn.pipeline import Pipeline  # para encadenar todos los pasos de FE en uno solo

## 2. Carga de datos

Leemos los ficheros parquet desde `../data_sample/`. Usamos `glob` con un patron comodin 
por si hay mas de un fichero parquet en esa carpeta.

In [2]:
# Recogemos el dataset completo.
df = pd.read_parquet('../data_sample/dataset_final.parquet') 

print(df.shape)
df.head()

(51001, 16)


,estacion,fecha,no2,temperatura,humedad,viento_vel,precipitacion,intensidad_mean,intensidad_max,ocupacion_mean,carga_mean,vmed_mean,vmed_max,tipo_elem_moda,distancia_meteo_km,distancia_trafico_km
0,4,2021-01-01,10,4.070833,NaN,NaN,NaN,388.155556,916.0,3.522222,21.722222,0.0,0.0,URB,0.0,0.06
1,4,2021-01-02,23,2.387500,NaN,NaN,NaN,516.344828,924.0,4.574713,28.563218,0.0,0.0,URB,0.0,0.06
2,4,2021-01-03,29,2.233333,NaN,NaN,NaN,476.076923,924.0,4.109890,26.329670,0.0,0.0,URB,0.0,0.06
3,4,2021-01-04,29,3.016667,NaN,NaN,NaN,677.088235,998.0,5.911765,37.044118,0.0,0.0,URB,0.0,0.06
4,4,2021-01-05,54,0.250000,NaN,NaN,NaN,588.095745,922.0,4.978723,32.202128,0.0,0.0,URB,0.0,0.06


## 3. Split cronologico: train_fe / test_fe

Este split es nuevo y va en paralelo al `train`/`test` aleatorio del notebook 02 (no lo sustituye).

**Por que un split por fecha y no aleatorio:** vamos a calcular lags y medias moviles de NO2 
(el valor de ayer, la media de los ultimos 7 dias...). Si el split fuera aleatorio, filas de 
fechas muy cercanas podrian caer una en train y otra en test, y un lag podria colarse informacion 
del futuro. Cortando por fecha, todo lo que hay en `test_fe` es estrictamente posterior a `train_fe`.

Usamos el percentil 80 de las fechas como corte, así queda aproximadamente un split 80/20.

In [3]:
# Fecha de corte: percentil 80 de las fechas disponibles (deja ~80/20)
fechas_unicas = sorted(df['fecha'].unique())
fecha_corte = fechas_unicas[int(len(fechas_unicas) * 0.8)]

train_fe = df[df['fecha'] < fecha_corte].copy()
test_fe = df[df['fecha'] >= fecha_corte].copy()

print(f"Fecha de corte: {fecha_corte}")
print(f"train_fe: {train_fe.shape[0]} filas ({train_fe['fecha'].min()} a {train_fe['fecha'].max()})")
print(f"test_fe:  {test_fe.shape[0]} filas ({test_fe['fecha'].min()} a {test_fe['fecha'].max()})")
print(f"Proporcion test_fe: {test_fe.shape[0] / df.shape[0]:.2%}")

Fecha de corte: 2023-09-25 00:00:00
train_fe: 40729 filas (2019-01-01 00:00:00 a 2023-09-24 00:00:00)
test_fe:  10272 filas (2023-09-25 00:00:00 a 2024-11-30 00:00:00)
Proporcion test_fe: 20.14%


## 4. Features temporales

Ampliamos el mes/dia de semana/finde que ya se vio en el notebook 03, con codificacion ciclica 
(seno/coseno) para que diciembre y enero queden "cerca" numericamente, y anadimos la estacion del anio.

Este transformer no aprende nada de los datos (no calcula ningun estadistico), solo deriva 
columnas nuevas a partir de la fecha. Por eso `fit` no hace nada.

In [ ]:
class FeaturesTemporales(BaseEstimator, TransformerMixin):
    # No aprende nada de los datos (no hay estadisticos que calcular),
    # asi que fit no hace nada y solo devuelve self, como exige la interfaz de sklearn
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()  # copia defensiva, para no modificar el DataFrame original por referencia

        # Codificacion ciclica del mes: en vez de 1-12, usamos seno/coseno
        # para que diciembre (12) y enero (1) queden "cerca" en el espacio numerico,
        # en vez de parecer los extremos opuestos de una escala lineal
        X['mes_sin'] = np.sin(2 * np.pi * X['fecha'].dt.month / 12)
        X['mes_cos'] = np.cos(2 * np.pi * X['fecha'].dt.month / 12)

        # Mismo truco pero con el dia del anio (1-365), da mas resolucion que el mes
        X['dia_anio_sin'] = np.sin(2 * np.pi * X['fecha'].dt.dayofyear / 365)
        X['dia_anio_cos'] = np.cos(2 * np.pi * X['fecha'].dt.dayofyear / 365)

        # Flag de fin de semana: dayofweek 5 y 6 son sabado y domingo
        X['finde'] = X['fecha'].dt.dayofweek.isin([5, 6]).astype(int)

        # Estacion del anio a partir del mes, usando np.select para evaluar
        # varias condiciones a la vez (mas legible que anidar np.where)
        mes = X['fecha'].dt.month
        X['estacion_anio'] = np.select(
            [mes.isin([12, 1, 2]), mes.isin([3, 4, 5]), mes.isin([6, 7, 8])],  # condiciones en orden
            ['invierno', 'primavera', 'verano'],  # valor si se cumple cada condicion
            default='otonio'  # lo que no entra en ninguna de las anteriores (9, 10, 11)
        )
        return X